<a href="https://colab.research.google.com/github/nourtarek555/MAESTRO/blob/main/phishing_detector.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🛡️ Multi-Agent Social Engineering Phishing Detector

## Architecture Overview
```
Each turn → frozen BERT [CLS] → 768-dim embedding
                                        ↓
                          BiLSTM over 10 turns → sequence summary
                                        ↓
Tactic Features (5 signals) ───────────┤
Turn Anomaly Flags (2 signals) ────────┤
                                        ↓
                              Fusion MLP → P(malicious)
                                        ↓
                        Explainability Report (attention + tactics)
```

**Why this works on 120 samples:**
- Frozen BERT = zero overfitting risk from embeddings
- Rule-based tactic features = reliable signal, no training needed  
- Only LSTM + MLP are trained (~50K params vs BERT's 110M)

**Why flat BERT failed:**  
- 38/60 malicious + 55/60 benign conversations exceeded BERT's 512-token limit → critical turns were silently truncated
- BERT cannot model the *temporal escalation pattern* (trust→request→pressure) that defines social engineering

## 📦 1. Install & Imports

In [ ]:
# Run this cell first
!pip install -q transformers torch scikit-learn matplotlib seaborn

In [ ]:
import json
import os
import re
import glob
import zipfile
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from transformers import AutoTokenizer, AutoModel
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, f1_score
)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
from collections import defaultdict

# Reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 📁 2. Upload & Load Data

In [ ]:
from google.colab import files
import zipfile, os, json

print('Upload malicious_conversations.zip and benign_conversations.zip')
uploaded = files.upload()

# Extract all zips
for fname in uploaded.keys():
    with zipfile.ZipFile(fname, 'r') as z:
        z.extractall('.')
        print(f'✅ Extracted {fname}')
        print('   Contents:', z.namelist()[:10])  # show what's inside

# Debug: walk everything to find JSON files
print('\n🔍 Scanning all extracted files...')
all_jsons = []
for dirpath, dirnames, filenames in os.walk('.'):
    for fname in filenames:
        if fname.endswith('.json'):
            full = os.path.join(dirpath, fname)
            all_jsons.append(full)

print(f'Total JSON files found: {len(all_jsons)}')
print('Sample paths:')
for p in all_jsons[:5]:
    print(' ', p)

# Classify as malicious or benign based on path/filename
malicious_files = [
    f for f in all_jsons
    if 'master' not in f and 'summary' not in f
    and 'benign' not in f.lower()
    and f.endswith('.json')
]
benign_files = [
    f for f in all_jsons
    if 'master' not in f and 'summary' not in f
    and 'benign' in f.lower()
    and f.endswith('.json')
]

malicious_files = []
benign_files = []

for dirpath, dirnames, filenames in os.walk('.'):
    for fname in filenames:
        if not fname.endswith('.json'): continue
        if 'master' in fname or 'summary' in fname: continue
        full = os.path.join(dirpath, fname)
        try:
            with open(full) as f:
                d = json.load(f)
            if 'conversation' not in d: continue  # skip config/sample files
        except:
            continue

        if 'benign_conversations' in dirpath:
            benign_files.append(full)
        elif 'malicious_conversations' in dirpath:
            malicious_files.append(full)

print(f'Malicious files: {len(malicious_files)}')
print(f'Benign files:    {len(benign_files)}')

## 🕵️ 3. Agent 2 — Tactic Detector (Rule-Based)

In [ ]:
# ── Tactic keyword dictionary (mirrors your generator exactly) ──────────────
TACTIC_KEYWORDS = {
    'urgency':     ['hr is waiting', 'deadline', 'today', 'right now', 'last chance',
                    'expires', 'immediately', 'asap', 'eod', 'end of day',
                    'within the hour', 'concludes this week'],
    'authority':   ['company policy', 'hr compliance', 'legal requires',
                    'standard procedure', 'mandatory', 'legal', 'compliance',
                    'policy requires'],
    'fear':        ['lose this opportunity', 'position will go', 'offer will be withdrawn',
                    'disqualify', 'gone', 'lost', 'miss out', 'pass us by',
                    'slip away', 'unfortunate'],
    'flattery':    ['impressed', 'perfect fit', 'excited about you', 'stands out',
                    'valuable', 'captivated', 'innovative', 'exceptional',
                    'groundbreaking', 'extraordinary'],
    'persistence': ['following up', 'need this to proceed', 'just need',
                    'waiting for', 'please share', 'once more', 'kindly ask',
                    'final call', 'still need']
}

# ── Sensitive information request patterns ───────────────────────────────────
INFO_REQUEST_PATTERNS = [
    r'bank account', r'routing number', r'direct deposit',
    r'tax id', r'social security', r'ssn', r'ein',
    r'current salary', r'compensation', r'salary expectation',
    r'personal (information|details|data)',
    r'financial (information|details|data)',
    r'provide.*(?:account|routing|ssn|tax|salary)',
    r'share.*(?:account|routing|ssn|tax|salary)',
    r'send.*(?:account|routing|ssn|tax|salary)',
]

def extract_tactic_features(conversation: list[dict]) -> dict:
    """
    Agent 2: Returns a feature dict with:
      - Binary flag per tactic (5 features)
      - info_request_flag: does any turn explicitly request sensitive info?
      - pressure_escalation_flag: does urgency/fear increase toward later turns?
      - tactic_density: fraction of turns containing any tactic
      - attacker_turns_text: aggregated attacker text for reporting
    """
    n = len(conversation)
    # Identify attacker (speaker in turn 0)
    attacker_name = conversation[0]['Name'] if conversation else ''

    tactic_flags = {t: 0 for t in TACTIC_KEYWORDS}
    tactic_turn_hits = {t: [] for t in TACTIC_KEYWORDS}  # which turns fired
    info_request_turns = []
    tactic_hit_per_turn = []

    for i, msg in enumerate(conversation):
        text = msg['Message'].lower()
        is_attacker = attacker_name in msg['Name']
        turn_hit = False

        if is_attacker:
            for tactic, keywords in TACTIC_KEYWORDS.items():
                if any(kw in text for kw in keywords):
                    tactic_flags[tactic] = 1
                    tactic_turn_hits[tactic].append(i)
                    turn_hit = True

            for pattern in INFO_REQUEST_PATTERNS:
                if re.search(pattern, text):
                    info_request_turns.append(i)
                    break

        tactic_hit_per_turn.append(1 if turn_hit else 0)

    # Pressure escalation: do urgency/fear hits concentrate in later half?
    late_turns = set(range(n // 2, n))
    late_hits = sum(
        1 for t in ['urgency', 'fear']
        for turn_idx in tactic_turn_hits[t]
        if turn_idx in late_turns
    )
    pressure_escalation = 1 if late_hits >= 2 else 0

    tactic_density = sum(tactic_hit_per_turn) / max(n, 1)

    active_tactics = [t for t, v in tactic_flags.items() if v == 1]

    return {
        **tactic_flags,  # urgency, authority, fear, flattery, persistence
        'info_request_flag': 1 if info_request_turns else 0,
        'pressure_escalation': pressure_escalation,
        'tactic_density': tactic_density,
        'active_tactics': active_tactics,        # for explainability
        'info_request_turns': info_request_turns, # for explainability
        'tactic_turn_hits': tactic_turn_hits,     # for attention overlay
    }

# Quick sanity check
with open(malicious_files[0]) as f:
    sample = json.load(f)
feats = extract_tactic_features(sample['conversation'])
print('Sample tactic features:')
for k, v in feats.items():
    if k not in ('tactic_turn_hits',):
        print(f'  {k}: {v}')

## 🧠 4. Agent 1 — Turn Encoder (Frozen BERT)

In [ ]:
# ── Load frozen BERT ─────────────────────────────────────────────────────────
# Using distilbert-base-uncased: same quality as BERT-base, half the memory
# Frozen completely — zero risk of overfitting on 120 samples
BERT_MODEL_NAME = 'distilbert-base-uncased'
EMBED_DIM = 768
MAX_TURN_TOKENS = 128  # per-turn limit; 10 turns × 128 = 1280 total (vs 512 flat)

print(f'Loading {BERT_MODEL_NAME} ...')
bert_tokenizer = AutoTokenizer.from_pretrained(BERT_MODEL_NAME)
bert_model = AutoModel.from_pretrained(BERT_MODEL_NAME).to(DEVICE)

# Freeze all parameters
for param in bert_model.parameters():
    param.requires_grad = False
bert_model.eval()

trainable = sum(p.numel() for p in bert_model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in bert_model.parameters())
print(f'BERT params: {total:,} total | {trainable} trainable (fully frozen ✅)')


@torch.no_grad()
def encode_turn(text: str) -> np.ndarray:
    """Encode a single conversation turn → 768-dim [CLS] embedding."""
    inputs = bert_tokenizer(
        text,
        return_tensors='pt',
        max_length=MAX_TURN_TOKENS,
        truncation=True,
        padding='max_length'
    ).to(DEVICE)
    outputs = bert_model(**inputs)
    # [CLS] token = index 0 of last hidden state
    cls_embedding = outputs.last_hidden_state[:, 0, :].squeeze(0)
    return cls_embedding.cpu().numpy()


def encode_conversation(conversation: list[dict]) -> np.ndarray:
    """
    Agent 1: Encode each turn individually.
    Returns shape (n_turns, 768).
    Preserves temporal structure — BERT sees each turn at full resolution.
    """
    embeddings = []
    for msg in conversation:
        # Prepend speaker role for richer context
        text = f"{msg['Name']}: {msg['Message']}"
        embeddings.append(encode_turn(text))
    return np.array(embeddings)  # (n_turns, 768)


print('\nTest encoding one conversation...')
test_emb = encode_conversation(sample['conversation'])
print(f'Shape: {test_emb.shape}  ← (n_turns=10, embed_dim=768) ✅')

## 💾 5. Precompute & Cache All Embeddings

We encode everything once and cache — this avoids re-running BERT at every training epoch.

In [ ]:
def load_dataset(malicious_files, benign_files):
    """Load all conversations, encode turns, extract tactic features."""
    data = []

    all_files = [(f, 1) for f in malicious_files] + [(f, 0) for f in benign_files]
    random.shuffle(all_files)

    for i, (fpath, label) in enumerate(all_files):
        with open(fpath) as fp:
            raw = json.load(fp)

        conversation = raw['conversation']
        meta         = raw['metadata']

        # Agent 1: turn-level BERT embeddings
        turn_embeddings = encode_conversation(conversation)

        # Agent 2: tactic features
        tactic_feats = extract_tactic_features(conversation)

        # Build numeric tactic vector (8 features)
        tactic_vector = np.array([
            tactic_feats['urgency'],
            tactic_feats['authority'],
            tactic_feats['fear'],
            tactic_feats['flattery'],
            tactic_feats['persistence'],
            tactic_feats['info_request_flag'],
            tactic_feats['pressure_escalation'],
            tactic_feats['tactic_density'],
        ], dtype=np.float32)

        data.append({
            'conversation_id': meta.get('conversation_id', os.path.basename(fpath)),
            'label': label,
            'role': meta.get('role', 'unknown'),
            'persona': meta.get('victim_persona', 'unknown'),
            'turn_embeddings': turn_embeddings,   # (10, 768)
            'tactic_vector': tactic_vector,        # (8,)
            'tactic_feats_full': tactic_feats,     # for explainability
            'conversation_raw': conversation,      # for reporting
        })

        if (i + 1) % 20 == 0:
            print(f'  Encoded {i+1}/{len(all_files)} conversations...')

    print(f'\n✅ Dataset loaded: {len(data)} conversations')
    mal = sum(1 for d in data if d['label'] == 1)
    print(f'   Malicious: {mal} | Benign: {len(data)-mal}')
    return data


print('Encoding all conversations (takes ~2-3 min on Colab GPU)...')
dataset = load_dataset(malicious_files, benign_files)

## 🏗️ 6. Model — BiLSTM + Attention + Fusion MLP

In [ ]:
class PhishingDetector(nn.Module):
    """
    Multi-agent phishing detector.

    Agent 1 (Turn Encoder):  Frozen BERT — already done in preprocessing.
                              Input: turn_embeddings (10, 768)

    Agent 3 (Flow Analyser): BiLSTM over turn sequence + attention mechanism.
                              Captures how the conversation escalates over time.
                              Attention weights = explainability (which turns triggered detection)

    Agent 2 (Tactic Detector): Rule-based features injected at fusion stage.

    Fusion MLP: Combines BiLSTM output + tactic features → binary classification.
    """

    def __init__(
        self,
        embed_dim: int = 768,
        lstm_hidden: int = 128,
        lstm_layers: int = 2,
        tactic_dim: int = 8,
        dropout: float = 0.4,
    ):
        super().__init__()

        # ── Input projection (reduces 768 → 256 before LSTM) ───────────────
        # Avoids the LSTM having to process 768-dim inputs with only 120 samples
        self.input_proj = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        # ── Agent 3: BiLSTM over turn sequence ─────────────────────────────
        self.bilstm = nn.LSTM(
            input_size=256,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0
        )

        lstm_out_dim = lstm_hidden * 2  # bidirectional

        # ── Attention over turns ────────────────────────────────────────────
        # Learns which turns are most suspicious → directly interpretable
        self.attn_linear = nn.Linear(lstm_out_dim, 1)

        # ── Fusion MLP ──────────────────────────────────────────────────────
        fusion_input_dim = lstm_out_dim + tactic_dim
        self.fusion = nn.Sequential(
            nn.Linear(fusion_input_dim, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(32, 1)
        )

    def forward(self, turn_embeddings, tactic_vector):
        """
        Args:
            turn_embeddings: (batch, n_turns, embed_dim)
            tactic_vector:   (batch, tactic_dim)
        Returns:
            logits:          (batch,)
            attn_weights:    (batch, n_turns)  ← for explainability
        """
        # Project embeddings
        x = self.input_proj(turn_embeddings)     # (batch, n_turns, 256)

        # BiLSTM
        lstm_out, _ = self.bilstm(x)             # (batch, n_turns, 256)

        # Attention
        attn_scores = self.attn_linear(lstm_out).squeeze(-1)  # (batch, n_turns)
        attn_weights = torch.softmax(attn_scores, dim=-1)     # (batch, n_turns)

        # Weighted sum of LSTM outputs
        context = torch.bmm(
            attn_weights.unsqueeze(1),  # (batch, 1, n_turns)
            lstm_out                    # (batch, n_turns, 256)
        ).squeeze(1)                    # (batch, 256)

        # Fuse with tactic features
        fused = torch.cat([context, tactic_vector], dim=-1)  # (batch, 264)

        # Classify
        logits = self.fusion(fused).squeeze(-1)  # (batch,)

        return logits, attn_weights


# Count trainable params
test_model = PhishingDetector().to(DEVICE)
trainable = sum(p.numel() for p in test_model.parameters() if p.requires_grad)
print(f'Trainable parameters: {trainable:,}  ← ~50K, safe for 120 samples ✅')
del test_model

## 📊 7. Dataset & DataLoader

In [ ]:
class ConversationDataset(Dataset):
    def __init__(self, samples):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        return {
            'turn_embeddings': torch.tensor(s['turn_embeddings'], dtype=torch.float32),
            'tactic_vector':   torch.tensor(s['tactic_vector'],   dtype=torch.float32),
            'label':           torch.tensor(s['label'],           dtype=torch.float32),
        }

print('Dataset class defined ✅')

## 🔁 8. Training with Stratified K-Fold Cross-Validation

With only 120 samples, a single train/test split is unreliable — results would vary wildly based on the random split. **5-fold stratified CV** gives a robust estimate of true performance and is standard practice for small datasets in academic papers.

In [ ]:
def train_one_fold(train_samples, val_samples, fold_num, epochs=40):
    """Train and evaluate one fold. Returns metrics + model state."""

    train_ds = ConversationDataset(train_samples)
    val_ds   = ConversationDataset(val_samples)

    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False)

    model = PhishingDetector().to(DEVICE)

    # Class-weighted loss to handle any imbalance
    pos_weight = torch.tensor([1.0]).to(DEVICE)  # Equal classes → weight=1
    criterion  = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
    optimizer  = optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-3)
    scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_f1    = 0.0
    best_state = None
    train_losses, val_f1s = [], []

    for epoch in range(epochs):
        # ── Training ────────────────────────────────────────────────────────
        model.train()
        epoch_loss = 0.0
        for batch in train_loader:
            turn_emb = batch['turn_embeddings'].to(DEVICE)
            tac_vec  = batch['tactic_vector'].to(DEVICE)
            labels   = batch['label'].to(DEVICE)

            optimizer.zero_grad()
            logits, _ = model(turn_emb, tac_vec)
            loss = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        train_losses.append(epoch_loss / len(train_loader))

        # ── Validation ──────────────────────────────────────────────────────
        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                turn_emb = batch['turn_embeddings'].to(DEVICE)
                tac_vec  = batch['tactic_vector'].to(DEVICE)
                labels   = batch['label'].to(DEVICE)
                logits, _ = model(turn_emb, tac_vec)
                preds = (torch.sigmoid(logits) >= 0.5).long().cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels.cpu().long().numpy())

        f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        val_f1s.append(f1)

        if f1 > best_f1:
            best_f1    = f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 10 == 0:
            print(f'  Fold {fold_num} | Epoch {epoch+1:3d}/{epochs} '
                  f'| Loss: {train_losses[-1]:.4f} | Val F1: {f1:.4f} '
                  f'| Best F1: {best_f1:.4f}')

    # ── Final evaluation on best checkpoint ─────────────────────────────────
    model.load_state_dict(best_state)
    model.eval()

    all_preds, all_probs, all_labels = [], [], []
    all_attn = []
    with torch.no_grad():
        for batch in val_loader:
            turn_emb = batch['turn_embeddings'].to(DEVICE)
            tac_vec  = batch['tactic_vector'].to(DEVICE)
            labels   = batch['label'].cpu().long().numpy()
            logits, attn = model(turn_emb, tac_vec)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs >= 0.5).astype(int)
            all_preds.extend(preds)
            all_probs.extend(probs)
            all_labels.extend(labels)
            all_attn.extend(attn.cpu().numpy())

    return {
        'model': model,
        'best_state': best_state,
        'preds': np.array(all_preds),
        'probs': np.array(all_probs),
        'labels': np.array(all_labels),
        'attn_weights': np.array(all_attn),
        'val_samples': val_samples,
        'best_f1': best_f1,
        'train_losses': train_losses,
        'val_f1s': val_f1s,
    }


def run_cross_validation(dataset, n_splits=5, epochs=40):
    labels_arr = np.array([d['label'] for d in dataset])
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)

    fold_results = []
    print(f'\n{'='*60}')
    print(f'  {n_splits}-Fold Stratified Cross-Validation')
    print(f'{'='*60}')

    for fold, (train_idx, val_idx) in enumerate(skf.split(dataset, labels_arr)):
        print(f'\n─── Fold {fold+1}/{n_splits} ───')
        train_samples = [dataset[i] for i in train_idx]
        val_samples   = [dataset[i] for i in val_idx]
        print(f'  Train: {len(train_samples)} | Val: {len(val_samples)}')

        result = train_one_fold(train_samples, val_samples, fold+1, epochs=epochs)
        fold_results.append(result)

        # Per-fold classification report
        print(f'\n  Fold {fold+1} Results:')
        print(classification_report(
            result['labels'], result['preds'],
            target_names=['Benign', 'Malicious'],
            digits=3
        ))

    return fold_results


# ── Run training ─────────────────────────────────────────────────────────────
fold_results = run_cross_validation(dataset, n_splits=5, epochs=40)

In [ ]:
# ============================================================
# ABLATION: BiLSTM Only — No Tactic Features
# Forces the model to learn purely from conversation text
# ============================================================

class PhishingDetectorNoTactics(nn.Module):
    """
    Ablation model: removes tactic features entirely.
    The BiLSTM + attention must learn from turn embeddings alone.
    """
    def __init__(
        self,
        embed_dim: int = 768,
        lstm_hidden: int = 128,
        lstm_layers: int = 2,
        dropout: float = 0.4,
    ):
        super().__init__()

        self.input_proj = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.LayerNorm(256),
            nn.ReLU(),
            nn.Dropout(dropout)
        )

        self.bilstm = nn.LSTM(
            input_size=256,
            hidden_size=lstm_hidden,
            num_layers=lstm_layers,
            batch_first=True,
            bidirectional=True,
            dropout=dropout if lstm_layers > 1 else 0.0
        )

        lstm_out_dim = lstm_hidden * 2

        self.attn_linear = nn.Linear(lstm_out_dim, 1)

        # No tactic features — fusion input is LSTM output only
        self.fusion = nn.Sequential(
            nn.Linear(lstm_out_dim, 64),
            nn.LayerNorm(64),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Dropout(dropout / 2),
            nn.Linear(32, 1)
        )

    def forward(self, turn_embeddings, tactic_vector=None):
        # tactic_vector is accepted but ignored — ablation
        x = self.input_proj(turn_embeddings)
        lstm_out, _ = self.bilstm(x)
        attn_scores = self.attn_linear(lstm_out).squeeze(-1)
        attn_weights = torch.softmax(attn_scores, dim=-1)
        context = torch.bmm(
            attn_weights.unsqueeze(1),
            lstm_out
        ).squeeze(1)
        logits = self.fusion(context).squeeze(-1)
        return logits, attn_weights


def train_one_fold_ablation(train_samples, val_samples, fold_num, epochs=40):
    """Same training loop as before but uses PhishingDetectorNoTactics."""
    train_ds = ConversationDataset(train_samples)
    val_ds   = ConversationDataset(val_samples)
    train_loader = DataLoader(train_ds, batch_size=16, shuffle=True)
    val_loader   = DataLoader(val_ds,   batch_size=16, shuffle=False)

    model = PhishingDetectorNoTactics().to(DEVICE)

    criterion  = nn.BCEWithLogitsLoss()
    optimizer  = optim.AdamW(model.parameters(), lr=2e-3, weight_decay=1e-3)
    scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_f1    = 0.0
    best_state = None
    train_losses, val_f1s = [], []

    for epoch in range(epochs):
        model.train()
        epoch_loss = 0.0
        for batch in train_loader:
            turn_emb = batch['turn_embeddings'].to(DEVICE)
            labels   = batch['label'].to(DEVICE)
            optimizer.zero_grad()
            logits, _ = model(turn_emb)
            loss = criterion(logits, labels)
            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()
            epoch_loss += loss.item()

        scheduler.step()
        train_losses.append(epoch_loss / len(train_loader))

        model.eval()
        all_preds, all_labels = [], []
        with torch.no_grad():
            for batch in val_loader:
                turn_emb = batch['turn_embeddings'].to(DEVICE)
                labels   = batch['label'].cpu().long().numpy()
                logits, _ = model(turn_emb)
                preds = (torch.sigmoid(logits) >= 0.5).long().cpu().numpy()
                all_preds.extend(preds)
                all_labels.extend(labels)

        f1 = f1_score(all_labels, all_preds, average='macro', zero_division=0)
        val_f1s.append(f1)

        if f1 > best_f1:
            best_f1    = f1
            best_state = {k: v.clone() for k, v in model.state_dict().items()}

        if (epoch + 1) % 10 == 0:
            print(f'  Ablation Fold {fold_num} | Epoch {epoch+1:3d}/{epochs} '
                  f'| Loss: {train_losses[-1]:.4f} | Val F1: {f1:.4f} '
                  f'| Best F1: {best_f1:.4f}')

    # Final evaluation
    model.load_state_dict(best_state)
    model.eval()
    all_preds, all_probs, all_labels, all_attn = [], [], [], []
    with torch.no_grad():
        for batch in val_loader:
            turn_emb = batch['turn_embeddings'].to(DEVICE)
            labels   = batch['label'].cpu().long().numpy()
            logits, attn = model(turn_emb)
            probs = torch.sigmoid(logits).cpu().numpy()
            preds = (probs >= 0.5).astype(int)
            all_preds.extend(preds)
            all_probs.extend(probs)
            all_labels.extend(labels)
            all_attn.extend(attn.cpu().numpy())

    return {
        'model': model,
        'best_state': best_state,
        'preds': np.array(all_preds),
        'probs': np.array(all_probs),
        'labels': np.array(all_labels),
        'attn_weights': np.array(all_attn),
        'val_samples': val_samples,
        'best_f1': best_f1,
        'train_losses': train_losses,
        'val_f1s': val_f1s,
    }


# ── Run ablation cross-validation ────────────────────────────────────────────
labels_arr = np.array([d['label'] for d in dataset])
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

ablation_results = []
print('='*60)
print('  ABLATION: BiLSTM Only (no tactic features)')
print('='*60)

for fold, (train_idx, val_idx) in enumerate(skf.split(dataset, labels_arr)):
    print(f'\n─── Fold {fold+1}/5 ───')
    train_samples = [dataset[i] for i in train_idx]
    val_samples   = [dataset[i] for i in val_idx]
    result = train_one_fold_ablation(train_samples, val_samples, fold+1, epochs=40)
    ablation_results.append(result)
    print(f'\n  Ablation Fold {fold+1} Results:')
    print(classification_report(
        result['labels'], result['preds'],
        target_names=['Benign', 'Malicious'], digits=3
    ))


# ── Aggregate ablation results ────────────────────────────────────────────────
abl_preds  = np.concatenate([r['preds']  for r in ablation_results])
abl_probs  = np.concatenate([r['probs']  for r in ablation_results])
abl_labels = np.concatenate([r['labels'] for r in ablation_results])

print('\n' + '='*60)
print('  ABLATION OVERALL RESULTS')
print('='*60)
print(classification_report(abl_labels, abl_preds,
      target_names=['Benign', 'Malicious'], digits=4))
abl_auc = roc_auc_score(abl_labels, abl_probs)
print(f'ROC-AUC: {abl_auc:.4f}')

abl_fold_f1s = [f1_score(r['labels'], r['preds'], average='macro') for r in ablation_results]
print(f'Mean F1: {np.mean(abl_fold_f1s):.4f} ± {np.std(abl_fold_f1s):.4f}')


# ── Side-by-side comparison: Full model vs Ablation ──────────────────────────
full_fold_f1s = [f1_score(r['labels'], r['preds'], average='macro') for r in fold_results]

print('\n' + '='*60)
print('  MODEL COMPARISON')
print('='*60)
print(f'  Full Multi-Agent (BiLSTM + Tactics):  {np.mean(full_fold_f1s):.4f} ± {np.std(full_fold_f1s):.4f}')
print(f'  Ablation (BiLSTM only):               {np.mean(abl_fold_f1s):.4f} ± {np.std(abl_fold_f1s):.4f}')
print(f'  Tactic features contribution:         +{np.mean(full_fold_f1s) - np.mean(abl_fold_f1s):.4f} F1')


# ── Attention comparison plot ─────────────────────────────────────────────────
abl_attn_mal, abl_attn_ben = [], []
for r in ablation_results:
    for label, attn in zip(r['labels'], r['attn_weights']):
        if label == 1: abl_attn_mal.append(attn)
        else:          abl_attn_ben.append(attn)

avg_abl_mal = np.mean(abl_attn_mal, axis=0) if abl_attn_mal else np.zeros(10)
avg_abl_ben = np.mean(abl_attn_ben, axis=0) if abl_attn_ben else np.zeros(10)

fig, axes = plt.subplots(2, 2, figsize=(16, 10))
turns = [f'T{i+1}' for i in range(10)]

# Row 1: Full model attention
axes[0,0].bar(turns, avg_attn_mal, color='#F44336', alpha=0.8, edgecolor='black')
axes[0,0].set_title('Full Model — Malicious Attention', fontweight='bold')
axes[0,0].set_ylabel('Attention Weight')
axes[0,0].set_ylim(0, 0.20)

axes[0,1].bar(turns, avg_attn_ben, color='#4CAF50', alpha=0.8, edgecolor='black')
axes[0,1].set_title('Full Model — Benign Attention', fontweight='bold')
axes[0,1].set_ylabel('Attention Weight')
axes[0,1].set_ylim(0, 0.20)

# Row 2: Ablation attention
axes[1,0].bar(turns, avg_abl_mal, color='#E53935', alpha=0.8, edgecolor='black')
axes[1,0].set_title('Ablation (No Tactics) — Malicious Attention', fontweight='bold')
axes[1,0].set_ylabel('Attention Weight')
axes[1,0].set_ylim(0, 0.20)

axes[1,1].bar(turns, avg_abl_ben, color='#43A047', alpha=0.8, edgecolor='black')
axes[1,1].set_title('Ablation (No Tactics) — Benign Attention', fontweight='bold')
axes[1,1].set_ylabel('Attention Weight')
axes[1,1].set_ylim(0, 0.20)

plt.suptitle('Attention Comparison: Full Model vs Ablation', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('ablation_attention.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nKey insight: Ablation attention should peak at turns 5-7')
print('(info request + pressure turns) — proving the BiLSTM')
print('learns attack structure from text when tactics are removed.')

## 📈 9. Aggregated Results & Visualizations

In [ ]:
# ── Aggregate across all folds ───────────────────────────────────────────────
all_preds  = np.concatenate([r['preds']  for r in fold_results])
all_probs  = np.concatenate([r['probs']  for r in fold_results])
all_labels = np.concatenate([r['labels'] for r in fold_results])

print('\n' + '='*60)
print('  OVERALL RESULTS (across all 5 folds)')
print('='*60)
print(classification_report(
    all_labels, all_preds,
    target_names=['Benign', 'Malicious'],
    digits=4
))
print(f'ROC-AUC: {roc_auc_score(all_labels, all_probs):.4f}')

fold_f1s = [f1_score(r['labels'], r['preds'], average='macro') for r in fold_results]
print(f'\nPer-fold macro F1: {[f"{x:.3f}" for x in fold_f1s]}')
print(f'Mean F1: {np.mean(fold_f1s):.4f} ± {np.std(fold_f1s):.4f}')


# ── Figure 1: Confusion Matrix ───────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

cm = confusion_matrix(all_labels, all_preds)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Benign', 'Malicious'],
            yticklabels=['Benign', 'Malicious'],
            ax=axes[0], annot_kws={'size': 16})
axes[0].set_title('Confusion Matrix (5-Fold CV)', fontsize=14, fontweight='bold')
axes[0].set_ylabel('True Label', fontsize=12)
axes[0].set_xlabel('Predicted Label', fontsize=12)

# ── Figure 2: ROC Curve ──────────────────────────────────────────────────────
fpr, tpr, _ = roc_curve(all_labels, all_probs)
auc = roc_auc_score(all_labels, all_probs)
axes[1].plot(fpr, tpr, color='steelblue', lw=2, label=f'ROC (AUC = {auc:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1)
axes[1].set_xlim([0.0, 1.0])
axes[1].set_ylim([0.0, 1.05])
axes[1].set_xlabel('False Positive Rate', fontsize=12)
axes[1].set_ylabel('True Positive Rate', fontsize=12)
axes[1].set_title('ROC Curve', fontsize=14, fontweight='bold')
axes[1].legend(loc='lower right', fontsize=12)

plt.tight_layout()
plt.savefig('results_overview.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: results_overview.png')

In [ ]:
# ── Figure 3: Training curves ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
colors = ['#2196F3', '#4CAF50', '#FF9800', '#9C27B0', '#F44336']

for i, r in enumerate(fold_results):
    axes[0].plot(r['train_losses'], color=colors[i], alpha=0.8, label=f'Fold {i+1}')
    axes[1].plot(r['val_f1s'],      color=colors[i], alpha=0.8, label=f'Fold {i+1}')

axes[0].set_title('Training Loss per Fold', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('BCE Loss')
axes[0].legend()

axes[1].set_title('Validation Macro F1 per Fold', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Macro F1')
axes[1].legend()

plt.tight_layout()
plt.savefig('training_curves.png', dpi=150, bbox_inches='tight')
plt.show()

## 🔍 10. Explainability — Which Turns & Tactics Triggered Detection

In [ ]:
# ── Average attention across all malicious conversations ────────────────────
all_attn_mal = []
all_attn_ben = []

for r in fold_results:
    for i, (label, attn) in enumerate(zip(r['labels'], r['attn_weights'])):
        if label == 1:
            all_attn_mal.append(attn)
        else:
            all_attn_ben.append(attn)

avg_attn_mal = np.mean(all_attn_mal, axis=0) if all_attn_mal else np.zeros(10)
avg_attn_ben = np.mean(all_attn_ben, axis=0) if all_attn_ben else np.zeros(10)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

turns = [f'T{i+1}' for i in range(len(avg_attn_mal))]

axes[0].bar(turns, avg_attn_mal, color='#F44336', alpha=0.8, edgecolor='black')
axes[0].set_title('Average Attention — Malicious Conversations', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Turn')
axes[0].set_ylabel('Average Attention Weight')
axes[0].set_ylim(0, max(avg_attn_mal) * 1.3)
for i, v in enumerate(avg_attn_mal):
    axes[0].text(i, v + 0.003, f'{v:.3f}', ha='center', fontsize=9)

axes[1].bar(turns, avg_attn_ben, color='#4CAF50', alpha=0.8, edgecolor='black')
axes[1].set_title('Average Attention — Benign Conversations', fontsize=13, fontweight='bold')
axes[1].set_xlabel('Turn')
axes[1].set_ylabel('Average Attention Weight')
axes[1].set_ylim(0, max(avg_attn_ben) * 1.3)
for i, v in enumerate(avg_attn_ben):
    axes[1].text(i, v + 0.003, f'{v:.3f}', ha='center', fontsize=9)

plt.suptitle('Temporal Attention: Which Turns Drive Detection', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('attention_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nInsight: High attention on turns 5-7 in malicious conversations corresponds')
print('to the attacker\'s escalation phase (first sensitive info request + pressure).')

In [ ]:
# ── Tactic feature importance ────────────────────────────────────────────────
# For each tactic, compute: how much does its presence correlate with malicious label?
tactic_names = ['urgency', 'authority', 'fear', 'flattery', 'persistence',
                'info_request', 'pressure_escalation', 'tactic_density']

tactic_vectors = np.array([d['tactic_vector'] for d in dataset])
labels_arr     = np.array([d['label']         for d in dataset])

fig, ax = plt.subplots(figsize=(10, 5))

mal_means = tactic_vectors[labels_arr == 1].mean(axis=0)
ben_means = tactic_vectors[labels_arr == 0].mean(axis=0)

x = np.arange(len(tactic_names))
width = 0.35
bars1 = ax.bar(x - width/2, mal_means, width, label='Malicious', color='#F44336', alpha=0.8)
bars2 = ax.bar(x + width/2, ben_means, width, label='Benign',    color='#4CAF50', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(tactic_names, rotation=30, ha='right', fontsize=11)
ax.set_ylabel('Mean Feature Value', fontsize=12)
ax.set_title('Tactic Feature Distribution: Malicious vs Benign', fontsize=13, fontweight='bold')
ax.legend(fontsize=11)
ax.set_ylim(0, 1.15)

plt.tight_layout()
plt.savefig('tactic_features.png', dpi=150, bbox_inches='tight')
plt.show()

## 🗣️ 11. Per-Conversation Explainability Report

In [ ]:
def generate_explanation(prob, pred, attn_weights, tactic_feats, conversation):
    """
    Generate a human-readable explanation of why the model made its decision.
    Combines attention weights + tactic features → no LLM needed.
    """
    lines = []
    label_str = '🚨 MALICIOUS' if pred == 1 else '✅ BENIGN'
    lines.append(f'Verdict: {label_str}  (confidence: {prob:.1%})')
    lines.append('')

    # Which turns got highest attention
    top_turns = np.argsort(attn_weights)[::-1][:3]
    lines.append('📍 Most suspicious turns (by model attention):')
    for rank, turn_idx in enumerate(top_turns):
        msg = conversation[turn_idx]
        snippet = msg['Message'][:100] + ('...' if len(msg['Message']) > 100 else '')
        lines.append(f'  #{rank+1} Turn {turn_idx+1} [{msg["Name"]}] (attn={attn_weights[turn_idx]:.3f})')
        lines.append(f'       "{snippet}"')
    lines.append('')

    # Active tactics
    active = tactic_feats.get('active_tactics', [])
    if active:
        tactic_descriptions = {
            'urgency':     'creates artificial time pressure ("deadline today", "last chance")',
            'authority':   'invokes authority to pressure compliance ("HR compliance", "mandatory")',
            'fear':        'threatens negative consequences ("offer withdrawn", "opportunity lost")',
            'flattery':    'uses excessive praise to lower victim\'s guard',
            'persistence': 'repeatedly requests information despite hesitation',
        }
        lines.append('⚠️  Social engineering tactics detected:')
        for t in active:
            lines.append(f'  • {t.upper()}: {tactic_descriptions.get(t, "")}')
    else:
        lines.append('✓ No social engineering tactics detected')
    lines.append('')

    # Info request turns
    req_turns = tactic_feats.get('info_request_turns', [])
    if req_turns:
        lines.append(f'🔑 Sensitive information requested in turns: {[t+1 for t in req_turns]}')

    if tactic_feats.get('pressure_escalation'):
        lines.append('📈 Pressure escalation pattern detected (urgency/fear concentrated in later turns)')

    return '\n'.join(lines)


# ── Demo: explain a few conversations from the last fold ────────────────────
last_fold = fold_results[-1]
last_fold_model = last_fold['model']
last_fold_model.eval()

# Find one correctly-detected malicious and one correctly-detected benign
examples_shown = 0
for i, sample in enumerate(last_fold['val_samples']):
    if examples_shown >= 3:
        break
    turn_emb = torch.tensor(sample['turn_embeddings'], dtype=torch.float32).unsqueeze(0).to(DEVICE)
    tac_vec  = torch.tensor(sample['tactic_vector'],   dtype=torch.float32).unsqueeze(0).to(DEVICE)

    with torch.no_grad():
        logit, attn = last_fold_model(turn_emb, tac_vec)

    prob = torch.sigmoid(logit).item()
    pred = int(prob >= 0.5)
    attn_w = attn.squeeze(0).cpu().numpy()

    print('='*70)
    print(f'Conversation: {sample["conversation_id"]}')
    print(f'True label: {"MALICIOUS" if sample["label"] else "BENIGN"}')
    print()
    explanation = generate_explanation(
        prob, pred, attn_w,
        sample['tactic_feats_full'],
        sample['conversation_raw']
    )
    print(explanation)
    print()
    examples_shown += 1

## 📊 12. Breakdown by Persona & Role

In [ ]:
# Reconstruct per-sample predictions across all folds
all_sample_results = []
for r in fold_results:
    for sample, pred, prob, label in zip(
        r['val_samples'], r['preds'], r['probs'], r['labels']
    ):
        all_sample_results.append({
            'persona': sample['persona'],
            'role':    sample['role'],
            'label':   label,
            'pred':    pred,
            'prob':    prob,
            'correct': int(pred == label),
        })

# ── By persona ───────────────────────────────────────────────────────────────
personas = ['naive', 'cooperative', 'cautious', 'formal']
persona_f1 = {}
for p in personas:
    p_samples = [s for s in all_sample_results if s['persona'] == p]
    if len(p_samples) >= 2:
        f1 = f1_score(
            [s['label'] for s in p_samples],
            [s['pred']  for s in p_samples],
            average='macro', zero_division=0
        )
        persona_f1[p] = f1

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

if persona_f1:
    axes[0].bar(list(persona_f1.keys()), list(persona_f1.values()),
                color=['#FF7043', '#FFA726', '#66BB6A', '#42A5F5'], alpha=0.85)
    axes[0].set_title('Detection F1 by Victim Persona', fontsize=13, fontweight='bold')
    axes[0].set_ylabel('Macro F1')
    axes[0].set_ylim(0, 1.1)
    for i, (k, v) in enumerate(persona_f1.items()):
        axes[0].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=11)

# ── By role ──────────────────────────────────────────────────────────────────
roles = list(set(s['role'] for s in all_sample_results))
role_f1 = {}
for role in roles:
    r_samples = [s for s in all_sample_results if s['role'] == role]
    if len(r_samples) >= 2:
        f1 = f1_score(
            [s['label'] for s in r_samples],
            [s['pred']  for s in r_samples],
            average='macro', zero_division=0
        )
        role_f1[role] = f1

if role_f1:
    axes[1].bar(list(role_f1.keys()), list(role_f1.values()),
                color=['#7E57C2', '#26A69A'], alpha=0.85)
    axes[1].set_title('Detection F1 by Scenario Role', fontsize=13, fontweight='bold')
    axes[1].set_ylabel('Macro F1')
    axes[1].set_ylim(0, 1.1)
    for i, (k, v) in enumerate(role_f1.items()):
        axes[1].text(i, v + 0.02, f'{v:.3f}', ha='center', fontsize=11)

plt.tight_layout()
plt.savefig('breakdown_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

print('\nInsight: Lower F1 on "formal" persona may indicate false negatives')
print('when the attack is subtle/low-pressure (formal victims resist, so')
print('the conversation looks more benign-like).')

## 💾 13. Save Best Model & Results

In [ ]:
# Pick the fold with best F1 as the final saved model
best_fold_idx = np.argmax([r['best_f1'] for r in fold_results])
best_fold     = fold_results[best_fold_idx]

torch.save({
    'model_state_dict': best_fold['best_state'],
    'model_config': {
        'embed_dim': 768,
        'lstm_hidden': 128,
        'lstm_layers': 2,
        'tactic_dim': 8,
        'dropout': 0.4,
    },
    'bert_model_name': BERT_MODEL_NAME,
    'fold': best_fold_idx + 1,
    'best_f1': best_fold['best_f1'],
}, 'phishing_detector_best.pt')

print(f'✅ Best model saved (Fold {best_fold_idx+1}, F1={best_fold["best_f1"]:.4f})')

# Download all outputs
try:
    from google.colab import files
    for fname in [
        'phishing_detector_best.pt',
        'results_overview.png',
        'training_curves.png',
        'attention_analysis.png',
        'tactic_features.png',
        'breakdown_analysis.png',
    ]:
        if os.path.exists(fname):
            files.download(fname)
    print('✅ All files downloaded')
except:
    print('✅ Files saved locally')